# 🎯 The Full Picture

## Evaluation + Tuning = The Software of the Future

We've seen each piece. Now let's put it all together:

- **Cross-model showdown** — how do different LLMs compare across tasks?
- **ROI of optimization** — is the tuning cost worth it?
- **Production patterns** — taking DSPy to prod

In [ ]:
import sys; sys.path.insert(0, ".")
import dspy, ipywidgets as widgets
from IPython.display import display, HTML
from dspy_tasks.tasks import get_task, list_tasks, list_by_tier, TASK_REGISTRY
from dspy_tasks.actions import run_baseline, run_optimization, compare_models
from dspy_tasks.visualize import *
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

MODELS = get_available_models()

## The Cross-Model Showdown

Let's run representative tasks from each tier against every model.
The heatmap reveals where models excel — and where they struggle.

In [ ]:
# Pick representative tasks from each tier
SHOWCASE_TASKS = ["sentiment", "math_word", "ticket_routing", "calculator_agent"]

btn = run_button("Run Full Showdown")
out = widgets.Output()

def on_showdown(b):
    with out:
        out.clear_output()
        task_names = []
        all_scores = []

        for task_id in SHOWCASE_TASKS:
            task = get_task(task_id)
            task_names.append(task.name)
            row = []

            for model in MODELS:
                print(f"  ⏳ {task.name} on {model.split('/')[-1]}...", end=" ")
                result = run_baseline(task_id, model, max_eval=5)
                row.append(result.score)
                print(f"{result.score:.0%}")

            all_scores.append(row)
            print()

        model_short = [m.split("/")[-1] for m in MODELS]
        fig = heatmap_tasks_models(task_names, model_short, all_scores,
            title="Task × Model Performance Matrix")
        fig.show()

btn.on_click(on_showdown)
display(btn, out)

## The ROI of Optimization

Optimization has a cost: extra LLM calls during the tuning phase.
But the benefit — higher accuracy on every future query — compounds.

Use the sliders below to explore the trade-off for your scenario.

In [ ]:
opt_calls = widgets.IntSlider(value=30, min=5, max=100, description="Optimization calls:")
cost_per = widgets.FloatSlider(value=0.003, min=0.001, max=0.01, step=0.001,
                                 description="$/call:", readout_format=".3f")
baseline_acc = widgets.FloatSlider(value=0.65, min=0.1, max=0.95, step=0.05, description="Baseline:")
optimized_acc = widgets.FloatSlider(value=0.88, min=0.1, max=0.99, step=0.05, description="Optimized:")
queries = widgets.IntSlider(value=10000, min=100, max=100000, step=1000, description="Queries:")

roi_out = widgets.Output()

def update_roi(*args):
    with roi_out:
        roi_out.clear_output()
        fig = cost_roi_chart(
            opt_calls.value, cost_per.value,
            baseline_acc.value, optimized_acc.value,
            queries.value, cost_per.value
        )
        fig.show()

for w in [opt_calls, cost_per, baseline_acc, optimized_acc, queries]:
    w.observe(update_roi, 'value')

display(widgets.VBox([opt_calls, cost_per, baseline_acc, optimized_acc, queries]), roi_out)
update_roi()  # initial render

## The New Software Stack

DSPy represents a paradigm shift. Here's how the pieces map:

| Traditional Software | AI Software (DSPy) |
|---|---|
| Source code | Signatures + Metrics + Data |
| Compiler | Optimizer (BootstrapFewShot / MIPROv2) |
| Binary | Optimized prompt + few-shot examples |
| Test suite | Evaluation dataset |
| CI/CD | Re-optimization pipeline |
| Refactoring | Re-compile with new data/model |

## Production Patterns

Taking DSPy from notebook to production:

1. **Save / Load** — Serialize optimized modules so optimization is a one-time cost
2. **Re-optimize on model change** — When you swap GPT-4o for a new model, re-run the optimizer
3. **Version your metrics** — Treat evaluation functions like source code (review, test, version)
4. **Monitor in production** — Log scores on live traffic to detect drift

In [ ]:
display_insight("Save & Load Optimized Modules",
    "In production, you save the optimized module with module.save('path.json') "
    "and load it with module.load('path.json'). The optimization cost is one-time; "
    "the benefit applies to every future query.",
    icon="💾")

---

## 🎯 The Punchline

In [ ]:
display(HTML('''
<div style="background: linear-gradient(135deg, #0078d4, #005a9e);
            color: white; padding: 32px; border-radius: 12px; margin: 20px 0; text-align: center">
    <h1 style="color: white; margin-bottom: 16px">🎯 The Punchline</h1>
    <p style="font-size: 1.3em; margin: 8px 0">
        <b>Evaluation</b> is the specification.
    </p>
    <p style="font-size: 1.3em; margin: 8px 0">
        <b>Optimization</b> is the compiler.
    </p>
    <p style="font-size: 1.3em; margin: 8px 0">
        <b>Data</b> is the source code.
    </p>
    <p style="font-size: 1.1em; margin-top: 20px; opacity: 0.9">
        Welcome to Software 3.0.
    </p>
</div>
'''))

---

## What's Next?

You've seen the full DSPy workflow. Here's where to go from here:

- 🔄 **Try with your own data** — Replace the ticket CSV with your domain data
- ➕ **Add new tasks** — Define a signature, write a metric, add examples
- 🧪 **Explore more optimizers** — Try MIPROv2, BootstrapFewShotWithRandomSearch
- 🚀 **Connect to production** — Point at your production LLM and optimize for real traffic

The key insight: **you don't write prompts anymore — you write specifications, and the optimizer writes the prompts for you.**